# 🩺 S6E7: Breaking the 0.950 Barrier [Complete Pipeline]

This kernel integrates the four advanced methodologies required to break through the 0.949 public leaderboard ceiling in Playground Series S6E7. 

Because the `health_condition` target is massively imbalanced (~86% `at-risk`, 8% `unhealthy`, 6% `fit`) and the competition is evaluated on **Balanced Accuracy**, standard out-of-the-box ML pipelines will fail. This notebook abandons traditional imputation and multi-class target chasing in favor of exact-value encodings, stratified local validation, and probability prior-correction.

---

## 1. Setup & Method 1: Zero Imputation (Native Missing Values)

Standard practice is to fill `NaN` values with the column mean or median. In this dataset, doing so destroys signal. The missing data configurations themselves hold predictive power. 

Instead of arbitrary imputation, we leave missing numerical values completely untouched. Modern gradient-boosted trees (LightGBM, CatBoost, XGBoost) possess native missing-value routing mechanisms. During training, they dynamically assign `NaN` values to whichever branch split minimizes the loss, effectively treating "missingness" as a native feature. For categoricals, we simply fill blanks with the string `"Missing"` so the ordinal encoder processes it as an isolated category.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd
import optuna
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import TargetEncoder, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

# ---------------------------------------------------------
# Load Data
# ---------------------------------------------------------
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')
sample_sub = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv')

X_train = train.drop(columns=['id', 'health_condition'])
y_train_raw = train['health_condition']
X_test = test.drop(columns=['id'])

# Map target to integers (0, 1, 2)
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)

cat_cols = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

# ---------------------------------------------------------
# Method 1: Do Not Impute Numerics
# ---------------------------------------------------------
# Isolate missing categoricals as their own group
X_train[cat_cols] = X_train[cat_cols].fillna('Missing')
X_test[cat_cols] = X_test[cat_cols].fillna('Missing')

# Encode standard categoricals to integers
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = oe.fit_transform(X_train[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])

---

## 2. Method 2: Exact-Value Target Encoding

Because this synthetic data is simulated from real wearable sensors, numerical features like `heart_rate` and `bmi` often contain discrete, repeating values rather than infinite floating points. 

Instead of binning these numbers, we apply **Target Encoding directly to the exact numerical values**. This converts each raw measurement into an array of probabilities (one for each class), allowing the trees to map complex, non-linear risk spikes that a standard `<` or `>` split would overlook. 

We use `cv=7` inside the `TargetEncoder` to safely cross-fit the data. This is critical to prevent catastrophic data leakage that would otherwise overfit the training set.

In [ ]:
print("Applying Exact-Value Target Encoding on Numerical Features...")

# Initialize multi-class target encoder with internal cross-fitting (cv=7)
te = TargetEncoder(target_type='multiclass', smooth='auto', cv=7)

# fit_transform internally cross-fits to prevent leakage
train_encoded_array = te.fit_transform(X_train[num_cols], y_train)

# Transform test data based on aggregate training statistics
test_encoded_array = te.transform(X_test[num_cols])

# Sklearn TargetEncoder creates multiple columns per original feature for multiclass
encoded_col_names = te.get_feature_names_out(num_cols)

df_train_enc = pd.DataFrame(train_encoded_array, columns=encoded_col_names, index=X_train.index)
df_test_enc = pd.DataFrame(test_encoded_array, columns=encoded_col_names, index=X_test.index)

# Replace raw numericals with their target-encoded probability equivalents
X_train = pd.concat([X_train.drop(columns=num_cols), df_train_enc], axis=1)
X_test = pd.concat([X_test.drop(columns=num_cols), df_test_enc], axis=1)

print(f"New Training Shape: {X_train.shape}")

---

## 3. Method 3: 7-Fold Stratified CV & Model Training

Because the `fit` class comprises less than 6% of the dataset, random train-test splitting will starve your validation sets. A strict **7-Fold Stratified Cross-Validation** forces every subset to maintain the exact 15:1 class ratio found in the overall dataset. 

We train three distinct, gradient-boosted architectures (LightGBM, CatBoost, XGBoost) to extract different predictive patterns from the encoded data. We use `class_weight='balanced'` where applicable to push the algorithms to care about the minority classes during tree building.

In [ ]:
N_SPLITS = 7
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Matrices to store Out-Of-Fold (OOF) and Test probabilities
oof_lgb = np.zeros((len(X_train), 3))
test_lgb = np.zeros((len(X_test), 3))

oof_cat = np.zeros((len(X_train), 3))
test_cat = np.zeros((len(X_test), 3))

oof_xgb = np.zeros((len(X_train), 3))
test_xgb = np.zeros((len(X_test), 3))

print("Initiating 7-Fold Stratified Training Ensemble...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
    
    # --- LightGBM ---
    clf_lgb = lgb.LGBMClassifier(
        objective='multiclass', class_weight='balanced', n_estimators=1200, 
        learning_rate=0.04, random_state=42, verbose=-1, n_jobs=-1
    )
    clf_lgb.fit(
        X_tr, y_tr, eval_set=[(X_va, y_va)], 
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    oof_lgb[val_idx] = clf_lgb.predict_proba(X_va)
    test_lgb += clf_lgb.predict_proba(X_test) / N_SPLITS
    
    # --- CatBoost ---
    clf_cat = cb.CatBoostClassifier(
        loss_function='MultiClass', auto_class_weights='Balanced', iterations=1200, 
        learning_rate=0.04, random_seed=42, verbose=0, thread_count=-1
    )
    clf_cat.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], early_stopping_rounds=50)
    oof_cat[val_idx] = clf_cat.predict_proba(X_va)
    test_cat += clf_cat.predict_proba(X_test) / N_SPLITS
    
    # --- XGBoost ---
    clf_xgb = xgb.XGBClassifier(
        objective='multi:softprob', n_estimators=1200, learning_rate=0.04, 
        random_state=42, tree_method='hist', enable_categorical=False, 
        n_jobs=-1, early_stopping_rounds=50
    )
    clf_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx] = clf_xgb.predict_proba(X_va)
    test_xgb += clf_xgb.predict_proba(X_test) / N_SPLITS

print("Model training complete.")

---

## 4. Method 4 (Part A): Mathematical Prior-Correction

This is the most critical step to maximize the Balanced Accuracy score. 

Even with internal class-weighting, the tree models will heavily over-predict the massive `at-risk` majority class. To correct this, we mathematically scale the output probabilities by dividing them by the dataset's class priors (raised to an exponent $\beta$). 

We use a grid search exclusively on our Out-Of-Fold (OOF) validation data to find the exact exponent that maximizes the metric. Once found, we apply those exact betas to correct the test set probabilities.

In [ ]:
def apply_prior_correction(probs, priors, beta):
    """Scales probability arrays against dataset class priors."""
    corrected_probs = probs / (priors ** beta)
    # Normalize row sums back to 1.0
    corrected_probs = corrected_probs / corrected_probs.sum(axis=1, keepdims=True)
    return corrected_probs

# 1. Calculate Exact Dataset Priors
class_counts = np.bincount(y_train)
priors = class_counts / len(y_train)
print(f"Calculated Dataset Priors: {np.round(priors, 4)}")

# 2. Grid Search Function for Beta
def tune_beta(oof_probs, y_true, priors):
    best_score, best_beta = 0, 0
    # Search grid from 0.0 (no correction) to 2.5 (aggressive correction)
    for beta in np.linspace(0.0, 2.5, 100):
        adj_probs = apply_prior_correction(oof_probs, priors, beta)
        preds = np.argmax(adj_probs, axis=1)
        score = balanced_accuracy_score(y_true, preds)
        if score > best_score:
            best_score, best_beta = score, beta
    return best_beta, best_score

# 3. Find optimal Betas on Out-Of-Fold Validation
beta_lgb, score_lgb = tune_beta(oof_lgb, y_train, priors)
beta_cat, score_cat = tune_beta(oof_cat, y_train, priors)
beta_xgb, score_xgb = tune_beta(oof_xgb, y_train, priors)

print(f"LGBM  - Opt Beta: {beta_lgb:.3f} | Score: {score_lgb:.5f}")
print(f"CatB  - Opt Beta: {beta_cat:.3f} | Score: {score_cat:.5f}")
print(f"XGB   - Opt Beta: {beta_xgb:.3f} | Score: {score_xgb:.5f}")

# 4. Apply the optimal Betas to correct both OOF and Test Probabilities
oof_lgb_corr = apply_prior_correction(oof_lgb, priors, beta_lgb)
test_lgb_corr = apply_prior_correction(test_lgb, priors, beta_lgb)

oof_cat_corr = apply_prior_correction(oof_cat, priors, beta_cat)
test_cat_corr = apply_prior_correction(test_cat, priors, beta_cat)

oof_xgb_corr = apply_prior_correction(oof_xgb, priors, beta_xgb)
test_xgb_corr = apply_prior_correction(test_xgb, priors, beta_xgb)

---

## 5. Method 4 (Part B): Optuna Probability Blending

Instead of arbitrarily guessing model weights (e.g., 40% / 30% / 30%), we use **Optuna** to optimize the blend. 

The objective function evaluates hundreds of weight combinations on the prior-corrected OOF probabilities, automatically finding the ratio that pushes the Balanced Accuracy metric to its absolute peak. Optuna ensures we are mathematically squeezing every possible fraction of a point out of our ensemble before generating the final submission file.

In [ ]:
# Mute Optuna's verbose trail logging
optuna.logging.set_verbosity(optuna.logging.WARNING)

def blending_objective(trial):
    # Suggest unnormalized weights
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_cat = trial.suggest_float('w_cat', 0.0, 1.0)
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    
    total_w = w_lgb + w_cat + w_xgb
    if total_w == 0:
        return 0.0
        
    # Normalize weights so they sum exactly to 1.0
    blended_oof = (
        oof_lgb_corr * (w_lgb / total_w) + 
        oof_cat_corr * (w_cat / total_w) + 
        oof_xgb_corr * (w_xgb / total_w)
    )
    
    preds = np.argmax(blended_oof, axis=1)
    return balanced_accuracy_score(y_train, preds)

# Run Optimization
study = optuna.create_study(direction="maximize")
study.optimize(blending_objective, n_trials=500)

# Extract Normalized Winning Weights
best_w = study.best_params
total_best_w = sum(best_w.values())

final_w_lgb = best_w['w_lgb'] / total_best_w
final_w_cat = best_w['w_cat'] / total_best_w
final_w_xgb = best_w['w_xgb'] / total_best_w

print(f"OOF Final Blended Score: {study.best_value:.5f}")
print(f"Optimal Weights -> LGBM: {final_w_lgb:.3f} | CatB: {final_w_cat:.3f} | XGB: {final_w_xgb:.3f}")

# ---------------------------------------------------------
# Final Prediction & Submission File Generation
# ---------------------------------------------------------
blended_test_probs = (
    test_lgb_corr * final_w_lgb + 
    test_cat_corr * final_w_cat + 
    test_xgb_corr * final_w_xgb
)

final_preds = np.argmax(blended_test_probs, axis=1)

# Inverse transform integers back to string labels
sample_sub['health_condition'] = le.inverse_transform(final_preds)
sample_sub.to_csv("submission.csv", index=False)
print("Pipeline successful. submission.csv generated!")